In [2]:
import cv2
import numpy as np
import os
import random
from pathlib import Path

# ================= 配置區域 =================
INPUT_DIR = "labelImg-master\\trainimg"
OUTPUT_DIR = "labelImg-master\\trainimg_fixed"
TARGET_SIZE = 1024
SCALE_RATIO = 0.7  # 草莓佔畫面的比例
PAD_COLOR = (114, 114, 114)
# ===========================================

def fix_and_verify():
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    img_exts = ('.jpg', '.jpeg', '.png', '.bmp')
    files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(img_exts)]
    
    print(f"🎯 修正模式啟動，預計處理 {len(files)} 張...")

    for filename in files:
        img_path = os.path.join(INPUT_DIR, filename)
        label_path = os.path.join(INPUT_DIR, Path(filename).stem + ".txt")
        
        # 1. 影像處理
        img = cv2.imread(img_path)
        if img is None: continue
        h_orig, w_orig = img.shape[:2]

        # 計算縮放率 (確保長邊縮小到 TARGET_SIZE * SCALE_RATIO)
        r = (TARGET_SIZE * SCALE_RATIO) / max(h_orig, w_orig)
        new_w, new_h = int(w_orig * r), int(h_orig * r)
        
        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
        canvas = np.full((TARGET_SIZE, TARGET_SIZE, 3), PAD_COLOR, dtype=np.uint8)
        
        # 計算偏移量 (置中)
        x_off = (TARGET_SIZE - new_w) // 2
        y_off = (TARGET_SIZE - new_h) // 2
        canvas[y_off:y_off+new_h, x_off:x_off+new_w] = resized
        cv2.imwrite(os.path.join(OUTPUT_DIR, filename), canvas)

        # 2. 標註座標修正 (核心邏輯)
        if os.path.exists(label_path):
            new_labels = []
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5: continue
                    cls, x, y, w, h = map(float, parts)
                    
                    # 關鍵：先還原到像素座標，縮放後再加偏移量，最後重新正規化
                    # YOLO x,y 是中心點
                    new_x_center = (x * w_orig * r + x_off) / TARGET_SIZE
                    new_y_center = (y * h_orig * r + y_off) / TARGET_SIZE
                    new_width = (w * w_orig * r) / TARGET_SIZE
                    new_height = (h * h_orig * r) / TARGET_SIZE
                    
                    new_labels.append(f"{int(cls)} {new_x_center:.6f} {new_y_center:.6f} {new_width:.6f} {new_height:.6f}")
            
            with open(os.path.join(OUTPUT_DIR, Path(filename).stem + ".txt"), 'w') as f:
                f.write("\n".join(new_labels))

    print(f"✨ 轉換完成！正在隨機抽取 3 張進行驗證...")
    verify_results()

def verify_results():
    """ 隨機抽樣並在畫面上畫出 Bbox，讓你肉眼確認 """
    img_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(('.jpg', '.png'))]
    samples = random.sample(img_files, min(3, len(img_files)))
    
    for s_name in samples:
        img = cv2.imread(os.path.join(OUTPUT_DIR, s_name))
        lab_path = os.path.join(OUTPUT_DIR, Path(s_name).stem + ".txt")
        
        if os.path.exists(lab_path):
            with open(lab_path, 'r') as f:
                for line in f:
                    cls, x, y, w, h = map(float, line.split())
                    # 轉回像素座標畫圖
                    x1 = int((x - w/2) * TARGET_SIZE)
                    y1 = int((y - h/2) * TARGET_SIZE)
                    x2 = int((x + w/2) * TARGET_SIZE)
                    y2 = int((y + h/2) * TARGET_SIZE)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2) # 綠色框
        
        # 顯示結果 (這會跳出視窗)
        cv2.imshow(f"Verify: {s_name}", img)
        print(f"正在展示 {s_name}，按任意鍵看下一張...")
        cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    fix_and_verify()

🎯 修正模式啟動，預計處理 400 張...
✨ 轉換完成！正在隨機抽取 3 張進行驗證...
正在展示 228.jpg，按任意鍵看下一張...
正在展示 153.jpg，按任意鍵看下一張...
正在展示 248.jpg，按任意鍵看下一張...
